# Y-Net Training and Test Evaluation

Run the notebook from top to bottom to train Y-Net, predict the test images, and calculate metrics. The saved 12-case comparison masks are read as a separate set; this notebook does not create replacements or export CSV/Excel files.


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from scipy import ndimage
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from IPython.display import display

def remove_small_srf(mask, min_pixels=10):
    labels, n = ndimage.label(mask == 2, structure=np.ones((3,3), dtype=np.uint8))
    sizes = np.bincount(labels.ravel())
    remove = np.zeros(n + 1, dtype=bool); remove[1:] = sizes[1:] < min_pixels
    mask[remove[labels]] = 0
    return mask

PROJECT = Path.cwd()
if PROJECT.name == "Code": PROJECT = PROJECT.parent
PACKAGE = PROJECT
DATA = PACKAGE / "Data"
REFERENCE = DATA / "test" / "masking_srf_reclassified"
REPLACED = DATA / "test" / "predictions_replaced_12_from_prior"
PALETTE = np.array([[0,0,0], [255,0,0], [0,255,0], [255,255,255]], dtype=np.uint8)
CLASSES = {"ONL": 1, "SRF": 2, "EZ-RPE": 3}
REGIONS = {"512": (0,512), "Central 1 mm": (216,296)}
SCALE_UM = 200 / 51
PIXEL_AREA_MM2 = 12.5 * SCALE_UM / 1_000_000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("Train / val / test:", 200, 33, 68)


## 1. Data preparation

Load the training, validation, and test images and masks.

In [ ]:
def crop_rgb(a):
    if a.shape[0] == 496:
        a = np.pad(a, ((8,8),(0,0),(0,0)), constant_values=0)
    if a.shape[:2] != (512,768):
        raise ValueError(f"Unexpected input size: {a.shape}")
    return a[:,128:640]

def label_image(path):
    m = crop_rgb(np.asarray(Image.open(path).convert("RGB")))
    y = np.full(m.shape[:2], 255, dtype=np.uint8)
    for i, color in enumerate(PALETTE): y[np.all(m == color, axis=2)] = i
    if np.any(y == 255): raise ValueError(f"Invalid mask palette color: {path}")
    return y

def load_xy(split, mask_folder):
    root = DATA / split
    origins = {p.stem: p for p in (root/"origin").glob("*") if p.is_file()}
    masks = {p.stem: p for p in (root/mask_folder).glob("*") if p.is_file()}
    if origins.keys() != masks.keys(): raise ValueError(f"{split}: OCT and mask filenames do not match")
    names = sorted(origins)
    xs = [torch.from_numpy(crop_rgb(np.asarray(Image.open(origins[n]).convert("RGB"))).copy()).permute(2,0,1) for n in names]
    ys = [torch.from_numpy(label_image(masks[n])) for n in names]
    return TensorDataset(torch.stack(xs), torch.stack(ys)), names

train_set, train_names = load_xy("train", "masking")
val_set, val_names = load_xy("val", "masking")
test_root = DATA / "test" / "origin"
test_names = sorted(p.name for p in test_root.glob("*.tif"))
print(f"Loaded: train={len(train_names)}, val={len(val_names)}, test={len(test_names)}")


## 2. Y-Net training

Train the segmentation model.

In [ ]:
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F


class FourierUnit(nn.Module):

    def __init__(self, in_channels, out_channels, groups=1, spatial_scale_factor=None, spatial_scale_mode='bilinear',
                 spectral_pos_encoding=False, use_se=False, se_kwargs=None, ffc3d=False, fft_norm='ortho'):
                           
        super(FourierUnit, self).__init__()
        self.groups = groups

        self.conv_layer = torch.nn.Conv2d(in_channels=in_channels * 2 + (2 if spectral_pos_encoding else 0),
                                          out_channels=out_channels * 2,
                                          kernel_size=1, stride=1, padding=0, groups=self.groups, bias=False)
        self.bn = torch.nn.BatchNorm2d(out_channels * 2)
        self.relu = torch.nn.ReLU(inplace=True)

                                      
        self.use_se = use_se
                   
                                  
                               
                                                                        

        self.spatial_scale_factor = spatial_scale_factor
        self.spatial_scale_mode = spatial_scale_mode
        self.spectral_pos_encoding = spectral_pos_encoding
        self.ffc3d = ffc3d
        self.fft_norm = fft_norm

    def forward(self, x):
        batch = x.shape[0]

        if self.spatial_scale_factor is not None:
            orig_size = x.shape[-2:]
            x = F.interpolate(x, scale_factor=self.spatial_scale_factor, mode=self.spatial_scale_mode, align_corners=False)

        r_size = x.size()
                                 
        fft_dim = (-3, -2, -1) if self.ffc3d else (-2, -1)
        ffted = torch.fft.rfftn(x, dim=fft_dim, norm=self.fft_norm)
        ffted = torch.stack((ffted.real, ffted.imag), dim=-1)
        clamp = False
        remove = False
        if clamp:
            ffted = torch.clamp(ffted, min=-10, max=10)
        if remove:
            fftedmin10 = torch.clamp(ffted, min=10)
            fftedmax10 = torch.clamp(ffted, max=-10)
            ffted = torch.where(ffted > 0, fftedmax10, fftedmin10)

        ffted = ffted.permute(0, 1, 4, 2, 3).contiguous()                           
        ffted = ffted.view((batch, -1,) + ffted.size()[3:])

        if self.spectral_pos_encoding:
            height, width = ffted.shape[-2:]
            coords_vert = torch.linspace(0, 1, height)[None, None, :, None].expand(batch, 1, height, width).to(ffted)
            coords_hor = torch.linspace(0, 1, width)[None, None, None, :].expand(batch, 1, height, width).to(ffted)
            ffted = torch.cat((coords_vert, coords_hor, ffted), dim=1)

                        
                                   

        ffted = self.conv_layer(ffted)                          
        ffted = self.relu(self.bn(ffted))

        ffted = ffted.view((batch, -1, 2,) + ffted.size()[2:]).permute(
            0, 1, 3, 4, 2).contiguous()                             
        ffted = torch.complex(ffted[..., 0], ffted[..., 1])

        ifft_shape_slice = x.shape[-3:] if self.ffc3d else x.shape[-2:]
        output = torch.fft.irfftn(ffted, s=ifft_shape_slice, dim=fft_dim, norm=self.fft_norm)

        if self.spatial_scale_factor is not None:
            output = F.interpolate(output, size=orig_size, mode=self.spatial_scale_mode, align_corners=False)

        return output

class SpectralTransform(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1, groups=1, enable_lfu=True, **fu_kwargs):
                           
        super(SpectralTransform, self).__init__()
        self.enable_lfu = enable_lfu
        if stride == 2:
            self.downsample = nn.AvgPool2d(kernel_size=(2, 2), stride=2)
        else:
            self.downsample = nn.Identity()

        self.stride = stride
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels //
                      2, kernel_size=1, groups=groups, bias=False),
            nn.BatchNorm2d(out_channels // 2),
            nn.ReLU(inplace=True)
        )
        self.fu = FourierUnit(
            out_channels // 2, out_channels // 2, groups, **fu_kwargs)
        if self.enable_lfu:
            self.lfu = FourierUnit(
                out_channels // 2, out_channels // 2, groups)
        self.conv2 = torch.nn.Conv2d(
            out_channels // 2, out_channels, kernel_size=1, groups=groups, bias=False)

    def forward(self, x):

        x = self.downsample(x)
        x = self.conv1(x)
        output = self.fu(x)

        if self.enable_lfu:
            n, c, h, w = x.shape
            split_no = 2
            split_s = h // split_no
            xs = torch.cat(torch.split(
                x[:, :c // 4], split_s, dim=-2), dim=1).contiguous()
            xs = torch.cat(torch.split(xs, split_s, dim=-1),
                           dim=1).contiguous()
            xs = self.lfu(xs)
            xs = xs.repeat(1, 1, split_no, split_no).contiguous()
        else:
            xs = 0

        output = self.conv2(x + output + xs)

        return output

class FFC(nn.Module):

    def __init__(self, in_channels, out_channels, kernel_size,
                 ratio_gin, ratio_gout, stride=1, padding=0,
                 dilation=1, groups=1, bias=False, enable_lfu=True,
                 padding_type='reflect', gated=False, **spectral_kwargs):
        super(FFC, self).__init__()

        assert stride == 1 or stride == 2, "Stride should be 1 or 2."
        self.stride = stride

        in_cg = int(in_channels * ratio_gin)
        in_cl = in_channels - in_cg
        out_cg = int(out_channels * ratio_gout)
        out_cl = out_channels - out_cg
                                                                  
                                                           

        self.ratio_gin = ratio_gin
        self.ratio_gout = ratio_gout
        self.global_in_num = in_cg

        module = nn.Identity if in_cl == 0 or out_cl == 0 else nn.Conv2d
        self.convl2l = module(in_cl, out_cl, kernel_size,
                              stride, padding, dilation, groups, bias, padding_mode=padding_type)
        module = nn.Identity if in_cl == 0 or out_cg == 0 else nn.Conv2d
        self.convl2g = module(in_cl, out_cg, kernel_size,
                              stride, padding, dilation, groups, bias, padding_mode=padding_type)
        module = nn.Identity if in_cg == 0 or out_cl == 0 else nn.Conv2d
        self.convg2l = module(in_cg, out_cl, kernel_size,
                              stride, padding, dilation, groups, bias, padding_mode=padding_type)
        module = nn.Identity if in_cg == 0 or out_cg == 0 else SpectralTransform
        self.convg2g = module(
            in_cg, out_cg, stride, 1 if groups == 1 else groups // 2, enable_lfu, **spectral_kwargs)

        self.gated = gated
        module = nn.Identity if in_cg == 0 or out_cl == 0 or not self.gated else nn.Conv2d
        self.gate = module(in_channels, 2, 1)

    def forward(self, x):
        x_l, x_g = x if type(x) is tuple else (x, 0)
        out_xl, out_xg = 0, 0

        if self.gated:
            total_input_parts = [x_l]
            if torch.is_tensor(x_g):
                total_input_parts.append(x_g)
            total_input = torch.cat(total_input_parts, dim=1)

            gates = torch.sigmoid(self.gate(total_input))
            g2l_gate, l2g_gate = gates.chunk(2, dim=1)
        else:
            g2l_gate, l2g_gate = 1, 1

        if self.ratio_gout != 1:
            if g2l_gate == 0:
                out_xl = self.convl2l(x_l)                                
            else:
                out_xl = self.convl2l(x_l) + self.convg2l(x_g) * g2l_gate
        if self.ratio_gout != 0:
            out_xg = self.convl2g(x_l) * l2g_gate + self.convg2g(x_g)

        return out_xl, out_xg

class FFC_BN_ACT(nn.Module):

    def __init__(self, in_channels, out_channels,
                 kernel_size, ratio_gin, ratio_gout,
                 stride=1, padding=0, dilation=1, groups=1, bias=False,
                 norm_layer=nn.BatchNorm2d, activation_layer=nn.Identity,
                 padding_type='reflect',
                 enable_lfu=True, **kwargs):
        super(FFC_BN_ACT, self).__init__()
        self.ffc = FFC(in_channels, out_channels, kernel_size,
                       ratio_gin, ratio_gout, stride, padding, dilation,
                       groups, bias, enable_lfu, padding_type=padding_type, **kwargs)
        lnorm = nn.Identity if ratio_gout == 1 else norm_layer
        gnorm = nn.Identity if ratio_gout == 0 else norm_layer
        global_channels = int(out_channels * ratio_gout)
        self.bn_l = lnorm(out_channels - global_channels)
        self.bn_g = gnorm(global_channels)

        lact = nn.Identity if ratio_gout == 1 else activation_layer
        gact = nn.Identity if ratio_gout == 0 else activation_layer
        self.act_l = lact(inplace=True)
        self.act_g = gact(inplace=True)

    def forward(self, x):
        x_l, x_g = self.ffc(x)
        x_l = self.act_l(self.bn_l(x_l))
        x_g = self.act_g(self.bn_g(x_g))
        return x_l, x_g

class ConcatTupleLayer(nn.Module):
    def forward(self, x):
        assert isinstance(x, tuple)
        x_l, x_g = x
        assert torch.is_tensor(x_l) or torch.is_tensor(x_g)
        if not torch.is_tensor(x_g):
            return x_l
        return torch.cat(x, dim=1)

class YNet_general(nn.Module):

    def __init__(self, in_channels=3, out_channels=1, init_features=32, ratio_in=0.5, ffc=True, skip_ffc=False,
                 cat_merge=True):
        super(YNet_general, self).__init__()

        self.ffc = ffc
        self.skip_ffc = skip_ffc
        self.ratio_in = ratio_in
        self.cat_merge = cat_merge

        features = init_features
                                                                  
        self.encoder1 = YNet_general._block(in_channels, features, name="enc1")
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = YNet_general._block(features, features * 2, name="enc2")           
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = YNet_general._block(features * 2, features * 4, name="enc3")
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = YNet_general._block(features * 4, features * 4, name="enc4")         
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        if ffc:
                                                                        
            self.encoder1_f = FFC_BN_ACT(in_channels, features, kernel_size=1, ratio_gin=0, ratio_gout=ratio_in)
            self.pool1_f = nn.MaxPool2d(kernel_size=2, stride=2)
            self.encoder2_f = FFC_BN_ACT(features, features * 2, kernel_size=1, ratio_gin=ratio_in,
                                         ratio_gout=ratio_in)           
            self.pool2_f = nn.MaxPool2d(kernel_size=2, stride=2)
            self.encoder3_f = FFC_BN_ACT(features * 2, features * 4, kernel_size=1, ratio_gin=ratio_in,
                                         ratio_gout=ratio_in)
            self.pool3_f = nn.MaxPool2d(kernel_size=2, stride=2)
            self.encoder4_f = FFC_BN_ACT(features * 4, features * 4, kernel_size=1, ratio_gin=ratio_in,
                                         ratio_gout=ratio_in)         
            self.pool4_f = nn.MaxPool2d(kernel_size=2, stride=2)

        else:
                                                                      
            self.encoder1_f = YNet_general._block(in_channels, features, name="enc1_2")
            self.pool1_f = nn.MaxPool2d(kernel_size=2, stride=2)
            self.encoder2_f = YNet_general._block(features, features * 2, name="enc2_2")           
            self.pool2_f = nn.MaxPool2d(kernel_size=2, stride=2)
            self.encoder3_f = YNet_general._block(features * 2, features * 4, name="enc3_2")   
            self.pool3_f = nn.MaxPool2d(kernel_size=2, stride=2)
            self.encoder4_f = YNet_general._block(features * 4, features * 4, name="enc4_2")         
            self.pool4_f = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = YNet_general._block(features * 8, features * 16, name="bottleneck")         

        if skip_ffc and not ffc:
            self.upconv4 = nn.ConvTranspose2d(
                features * 16, features * 8, kernel_size=2, stride=2      
            )
            self.decoder4 = YNet_general._block((features * 8) * 2, features * 8, name="dec4")         
            self.upconv3 = nn.ConvTranspose2d(
                features * 8, features * 4, kernel_size=2, stride=2
            )
            self.decoder3 = YNet_general._block((features * 6) * 2, features * 4, name="dec3")
            self.upconv2 = nn.ConvTranspose2d(
                features * 4, features * 2, kernel_size=2, stride=2
            )
            self.decoder2 = YNet_general._block((features * 3) * 2, features * 2, name="dec2")
            self.upconv1 = nn.ConvTranspose2d(
                features * 2, features, kernel_size=2, stride=2
            )
            self.decoder1 = YNet_general._block(features * 3, features, name="dec1")       

        elif skip_ffc and ffc:
            self.upconv4 = nn.ConvTranspose2d(
                features * 16, features * 8, kernel_size=2, stride=2      
            )
            self.decoder4 = YNet_general._block((features * 8) * 2, features * 8, name="dec4")         
            self.upconv3 = nn.ConvTranspose2d(
                features * 8, features * 4, kernel_size=2, stride=2
            )
            self.decoder3 = YNet_general._block((features * 6) * 2, features * 4, name="dec3")
            self.upconv2 = nn.ConvTranspose2d(
                features * 4, features * 2, kernel_size=2, stride=2
            )
            self.decoder2 = YNet_general._block((features * 3) * 2, features * 2, name="dec2")
            self.upconv1 = nn.ConvTranspose2d(
                features * 2, features, kernel_size=2, stride=2
            )
            self.decoder1 = YNet_general._block(features * 3, features, name="dec1")       

        else:
            self.upconv4 = nn.ConvTranspose2d(
                features * 16, features * 8, kernel_size=2, stride=2      
            )
            self.decoder4 = YNet_general._block((features * 6) * 2, features * 8, name="dec4")         
            self.upconv3 = nn.ConvTranspose2d(
                features * 8, features * 4, kernel_size=2, stride=2
            )
            self.decoder3 = YNet_general._block((features * 4) * 2, features * 4, name="dec3")
            self.upconv2 = nn.ConvTranspose2d(
                features * 4, features * 2, kernel_size=2, stride=2
            )
            self.decoder2 = YNet_general._block((features * 2) * 2, features * 2, name="dec2")
            self.upconv1 = nn.ConvTranspose2d(
                features * 2, features, kernel_size=2, stride=2
            )
            self.decoder1 = YNet_general._block(features * 2, features, name="dec1")       

        self.conv = nn.Conv2d(
            in_channels=features, out_channels=out_channels, kernel_size=1
        )
        self.softmax = nn.Softmax2d()
        self.catLayer = ConcatTupleLayer()

    def apply_fft(self, inp, batch):
        ffted = torch.fft.fftn(inp)
        ffted = torch.stack((ffted.real, ffted.imag), dim=-1)

        ffted = ffted.permute(0, 1, 4, 2, 3).contiguous()
        ffted = ffted.view((batch, -1,) + ffted.size()[3:])
        return ffted

    def forward(self, x):
        batch = x.shape[0]
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))

        enc3 = self.encoder3(self.pool2(enc2))

        enc4 = self.encoder4(self.pool3(enc3))
        enc4_2 = self.pool4(enc4)

        if self.ffc:
            enc1_f = self.encoder1_f(x)
            enc1_l, enc1_g = enc1_f
            if self.ratio_in == 0:
                enc2_f = self.encoder2_f((self.pool1_f(enc1_l), enc1_g))
            elif self.ratio_in == 1:
                enc2_f = self.encoder2_f((enc1_l, self.pool1_f(enc1_g)))
            else:
                enc2_f = self.encoder2_f((self.pool1_f(enc1_l), self.pool1_f(enc1_g)))

            enc2_l, enc2_g = enc2_f
            if self.ratio_in == 0:
                enc3_f = self.encoder3_f((self.pool2_f(enc2_l), enc2_g))
            elif self.ratio_in == 1:
                enc3_f = self.encoder3_f((enc2_l, self.pool2_f(enc2_g)))
            else:
                enc3_f = self.encoder3_f((self.pool2_f(enc2_l), self.pool2_f(enc2_g)))

            enc3_l, enc3_g = enc3_f
            if self.ratio_in == 0:
                enc4_f = self.encoder4_f((self.pool3_f(enc3_l), enc3_g))
            elif self.ratio_in == 1:
                enc4_f = self.encoder4_f((enc3_l, self.pool3_f(enc3_g)))
            else:
                enc4_f = self.encoder4_f((self.pool3_f(enc3_l), self.pool3_f(enc3_g)))

            enc4_l, enc4_g = enc4_f
            if self.ratio_in == 0:
                enc4_f2 = self.pool1_f(enc4_l)
            elif self.ratio_in == 1:
                enc4_f2 = self.pool1_f(enc4_g)
            else:
                enc4_f2 = self.catLayer((self.pool4_f(enc4_l), self.pool4_f(enc4_g)))

        else:
            enc1_f = self.encoder1_f(x)
            enc2_f = self.encoder2_f(self.pool1_f(enc1_f))
            enc3_f = self.encoder3_f(self.pool2_f(enc2_f))
            enc4_f = self.encoder4_f(self.pool3_f(enc3_f))
            enc4_f2 = self.pool4(enc4_f)

        if self.cat_merge:
            a = torch.zeros_like(enc4_2)
            b = torch.zeros_like(enc4_f2)

            enc4_2 = enc4_2.view(torch.numel(enc4_2), 1)
            enc4_f2 = enc4_f2.view(torch.numel(enc4_f2), 1)

            bottleneck = torch.cat((enc4_2, enc4_f2), 1)
            bottleneck = bottleneck.view_as(torch.cat((a, b), 1))

        else:
            bottleneck = torch.cat((enc4_2, enc4_f2), 1)

        bottleneck = self.bottleneck(bottleneck)

        dec4 = self.upconv4(bottleneck)

        if self.ffc and self.skip_ffc:
            enc4_in = torch.cat((enc4, self.catLayer((enc4_f[0], enc4_f[1]))), dim=1)

            dec4 = torch.cat((dec4, enc4_in), dim=1)
            dec4 = self.decoder4(dec4)
            dec3 = self.upconv3(dec4)

            enc3_in = torch.cat((enc3, self.catLayer((enc3_f[0], enc3_f[1]))), dim=1)
            dec3 = torch.cat((dec3, enc3_in), dim=1)
            dec3 = self.decoder3(dec3)

            dec2 = self.upconv2(dec3)
            enc2_in = torch.cat((enc2, self.catLayer((enc2_f[0], enc2_f[1]))), dim=1)
            dec2 = torch.cat((dec2, enc2_in), dim=1)
            dec2 = self.decoder2(dec2)
            dec1 = self.upconv1(dec2)
            enc1_in = torch.cat((enc1, self.catLayer((enc1_f[0], enc1_f[1]))), dim=1)
            dec1 = torch.cat((dec1, enc1_in), dim=1)

        elif self.skip_ffc:
            enc4_in = torch.cat((enc4, enc4_f), dim=1)

            dec4 = torch.cat((dec4, enc4_in), dim=1)
            dec4 = self.decoder4(dec4)
            dec3 = self.upconv3(dec4)

            enc3_in = torch.cat((enc3, enc3_f), dim=1)
            dec3 = torch.cat((dec3, enc3_in), dim=1)
            dec3 = self.decoder3(dec3)

            dec2 = self.upconv2(dec3)
            enc2_in = torch.cat((enc2, enc2_f), dim=1)
            dec2 = torch.cat((dec2, enc2_in), dim=1)
            dec2 = self.decoder2(dec2)
            dec1 = self.upconv1(dec2)
            enc1_in = torch.cat((enc1, enc1_f), dim=1)
            dec1 = torch.cat((dec1, enc1_in), dim=1)

        else:
            dec4 = torch.cat((dec4, enc4), dim=1)
            dec4 = self.decoder4(dec4)
            dec3 = self.upconv3(dec4)
            dec3 = torch.cat((dec3, enc3), dim=1)
            dec3 = self.decoder3(dec3)
            dec2 = self.upconv2(dec3)
            dec2 = torch.cat((dec2, enc2), dim=1)
            dec2 = self.decoder2(dec2)
            dec1 = self.upconv1(dec2)
            dec1 = torch.cat((dec1, enc1), dim=1)

        dec1 = self.decoder1(dec1)

        return self.conv(dec1)

    @staticmethod
    def _block(in_channels, features, name):
        return nn.Sequential(
            OrderedDict(
                [
                    (
                        name + "conv1",
                        nn.Conv2d(
                            in_channels=in_channels,
                            out_channels=features,
                            kernel_size=3,
                            padding=1,
                            bias=False,
                        ),
                    ),
                    (name + "norm1", nn.BatchNorm2d(num_features=features)),
                    (name + "relu1", nn.ReLU(inplace=True)),
                    (
                        name + "conv2",
                        nn.Conv2d(
                            in_channels=features,
                            out_channels=features,
                            kernel_size=3,
                            padding=1,
                            bias=False,
                        ),
                    ),
                    (name + "norm2", nn.BatchNorm2d(num_features=features)),
                    (name + "relu2", nn.ReLU(inplace=True)),
                ]
            )
        )


In [ ]:
def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def loss_fn(logits, target):
    ce = nn.functional.cross_entropy(logits, target)
    prob = logits.softmax(1)[:,1:]
    truth = nn.functional.one_hot(target, 4).permute(0,3,1,2)[:,1:].to(prob.dtype)
    inter = (prob*truth).sum((2,3)); den = prob.sum((2,3))+truth.sum((2,3))
    return ce + 1 - ((2*inter+1e-6)/(den+1e-6)).mean()

seed_all()
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
torch.set_num_threads(8)
model = YNet_general(out_channels=4).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=5, threshold=1e-3, threshold_mode="rel")
generator = torch.Generator().manual_seed(42)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True, generator=generator, num_workers=0, pin_memory=DEVICE.type=="cuda")
val_loader = DataLoader(val_set, batch_size=8, shuffle=False, num_workers=0, pin_memory=DEVICE.type=="cuda")
best_val, best_epoch, stale = float("inf"), 0, 0
best_weights = None
history = []
for epoch in range(1,101):
    model.train(); train_sum = train_n = 0
    for x,y in train_loader:
        x=x.to(DEVICE).float()/255; y=y.to(DEVICE).long()
        optimizer.zero_grad(set_to_none=True); loss=loss_fn(model(x),y); loss.backward(); optimizer.step()
        train_sum += loss.item()*len(x); train_n += len(x)
    model.eval(); val_sum = val_n = 0
    with torch.inference_mode():
        for x,y in val_loader:
            x=x.to(DEVICE).float()/255; y=y.to(DEVICE).long()
            val_sum += loss_fn(model(x),y).item()*len(x); val_n += len(x)
    train_loss, val_loss = train_sum/train_n, val_sum/val_n
    scheduler.step(val_loss)
    if val_loss < best_val:
        best_val, best_epoch, stale = val_loss, epoch, 0
        best_weights = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else: stale += 1
    history.append({"epoch":epoch,"train_loss":train_loss,"val_loss":val_loss,"best_val_loss":best_val})
    print(f"{epoch:03d} | train {train_loss:.5f} | val {val_loss:.5f} | best {best_val:.5f} (epoch {best_epoch})")
    if stale >= 30: break
model.load_state_dict(best_weights)
model.eval()
print(f"Training complete: {len(history)} epochs, best epoch={best_epoch}, best val loss={best_val:.5f}")


## 3. Test inference

Predict segmentation masks for the test images.

In [ ]:
test_x = torch.stack([torch.from_numpy(crop_rgb(np.asarray(Image.open(test_root/n).convert("RGB"))).copy()).permute(2,0,1) for n in test_names])
test_loader = DataLoader(TensorDataset(test_x), batch_size=8, shuffle=False, num_workers=0)
new_predictions = {}
with torch.inference_mode():
    offset = 0
    for (x,) in test_loader:
        pred = model(x.to(DEVICE).float()/255).argmax(1).cpu().numpy()
        for arr in pred: new_predictions[test_names[offset]] = arr.astype(np.uint8); offset += 1
print("Test inference complete:", len(new_predictions), "images")
new_predictions = {name: remove_small_srf(mask) for name, mask in new_predictions.items()}


## 4. Image-level metrics

Calculate Dice, thickness/MAE, and SRF area for new predictions and the saved 12-case comparison.

In [ ]:
def rgb_to_labels(rgb):
    out=np.full(rgb.shape[:2],255,dtype=np.uint8)
    for i,color in enumerate(PALETTE): out[np.all(rgb==color,axis=2)]=i
    if np.any(out==255): raise ValueError("Mask contains an unknown color")
    return out

def dice(a,b):
    den=int(a.sum()+b.sum())
    return 2*int(np.count_nonzero(a&b))/den if den else 1.0

def measure_predictions(source, pred_map):
    result=[]
    for name in test_names:
        ref_rgb=crop_rgb(np.asarray(Image.open(REFERENCE/(Path(name).stem+".tif")).convert("RGB")))
        ref=rgb_to_labels(ref_rgb)
        pred=pred_map[name]
        group="SRF present" if np.any(ref==2) else "SRF absent"
        for region,(x1,x2) in REGIONS.items():
            for label,cid in CLASSES.items():
                a=ref[:,x1:x2]==cid; b=pred[:,x1:x2]==cid
                row={"source":source,"image":name,"SRF_group":group,"region":region,"class":label,"Dice":dice(a,b)}
                if cid==2:
                    row["reference_area_mm2"]=a.sum()*PIXEL_AREA_MM2
                    row["prediction_area_mm2"]=b.sum()*PIXEL_AREA_MM2
                else:
                    row["reference_thickness_um"]=a.sum()/(x2-x1)*SCALE_UM
                    row["prediction_thickness_um"]=b.sum()/(x2-x1)*SCALE_UM
                    row["MAE_um"]=abs(row["reference_thickness_um"]-row["prediction_thickness_um"])
                result.append(row)
    return pd.DataFrame(result)

new_metrics = measure_predictions("Newly trained Y-Net", new_predictions)
replaced_predictions = {name: rgb_to_labels(np.asarray(Image.open(REPLACED/(Path(name).stem+".png")).convert("RGB"))) for name in test_names}
replaced_metrics = measure_predictions("Saved 12-case replacement", replaced_predictions)
all_metrics = pd.concat([new_metrics,replaced_metrics], ignore_index=True)
summary = (all_metrics.groupby(["source","SRF_group","region","class"], sort=False, dropna=False)
    .agg(n_images=("image","nunique"), mean_Dice=("Dice","mean"),
         reference_thickness_um=("reference_thickness_um","mean"), prediction_thickness_um=("prediction_thickness_um","mean"), MAE_um=("MAE_um","mean"),
         reference_SRF_area_mm2=("reference_area_mm2","mean"), prediction_SRF_area_mm2=("prediction_area_mm2","mean"))
    .reset_index())


## 5. Display

Show grouped summaries and image-level results.

In [ ]:
display(summary.round(4))
print("Image-level metrics:", len(all_metrics), "rows")
display(all_metrics.round(4))
